# 03 · ข้อ 2 ต้นปี 2569 เทียบช่วงเดียวกันของ 2568
ผู้รับผิดชอบ: สมาชิก B · ต้องรัน `01_cleaning.ipynb` ก่อน

**คำถาม:** ปริมาณผู้ใช้บริการช่วงต้นปี พ.ศ. 2569 เปลี่ยนแปลงจากช่วงเดียวกันของปี พ.ศ. 2568 อย่างไร และวิธีจัดการค่าว่าง/ค่าศูนย์ต่างกันส่งผลต่ออันดับหรือข้อสรุปอย่างไร

โครงของ notebook ตรงกับหัวข้อที่รายงานต้องมีในแต่ละข้อ: Feature Selection → Data Preparation → Feature Engineering → Visualization → Analysis & Findings

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import prep

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
prep.setup_thai_font()

'Loma'

In [2]:
df = prep.load_clean()
catalog = pd.read_csv(prep.PREPARED_DIR / "service_catalog.csv", encoding="utf-8-sig", keep_default_na=False)
print(df.shape, df["year_be"].value_counts().to_dict())

(18705, 18) {2568: 15695, 2569: 3010}


## 1. Feature Selection

| Feature ต้นฉบับ | เหตุผล |
|---|---|
| `date`, `year_be` | กำหนดช่วงเทียบที่ตรงกันของสองปี |
| `service` | หน่วยที่เปรียบเทียบ |
| `value`, `missing_flag`, `zero_flag`, `zero_long_run` | จำนวน และหลักฐานสำหรับทดสอบวิธีจัดการค่าว่าง/ศูนย์ |
| `unit` | ใช้ “คน” เป็นหลัก (“คัน” แยกแสดงเป็นตัวอย่างที่เทียบไม่ได้) |

## 2. Data Preparation

In [3]:
# ช่วงเทียบ: 1 ม.ค. → วันสุดท้ายที่มีข้อมูลปี 2569 (คำนวณจากข้อมูล ไม่ฮาร์ดโค้ด)
last_69 = df.loc[df["year_be"] == 2569, "date"].max()
end_md = (last_69.month, last_69.day)

def in_window(d):
    return (d.dt.month < end_md[0]) | ((d.dt.month == end_md[0]) & (d.dt.day <= end_md[1]))

# เก็บบริการ "คน" ที่ไม่ใช่ยอดรวม (ยังไม่ตัด all-zero/ศูนย์ช่วงยาว เพราะต้องใช้ทดสอบ S1–S5)
q2 = df[(df["unit"] == "คน") & (~df["is_aggregate"]) & in_window(df["date"])].copy()
win = q2.groupby("year_be")["date"].agg(["min", "max", "nunique"])
win["weekend_days"] = q2.drop_duplicates(["year_be", "date"]).groupby("year_be")["date"].apply(lambda s: (s.dt.dayofweek >= 5).sum())
display(win)
print(q2["service"].nunique(), "services")

,min,max,nunique,weekend_days
year_be,,,,
2568,2025-01-01,2025-03-11,70,20
2569,2026-01-01,2026-03-11,70,20


39 services


> ต้องบันทึกอีกไฟล์: `q2_missing_sensitivity.csv` (yoy_change_pct และอันดับของทุกบริการในแต่ละสถานการณ์ S1–S5)

## 3. Feature Engineering (อย่างน้อย 3 ตัวแปร)

| Feature ใหม่ | สูตร / เงื่อนไข | ประโยชน์ |
|---|---|---|
| `avg_daily_68`, `avg_daily_69` | ค่าเฉลี่ยต่อวันในช่วงเทียบ (หารด้วยวันที่มีข้อมูลจริง) | เทียบได้แม้จำนวนวันที่มีข้อมูลไม่เท่ากัน |
| `yoy_change_pct` | `(avg_daily_69 − avg_daily_68) / avg_daily_68 × 100` | คำตอบหลัก |
| `abs_change` | `avg_daily_69 − avg_daily_68` | บริการเล็กอาจมี % สูงแต่จำนวนน้อย |
| `coverage_ratio` | วันที่มีข้อมูลใช้ได้ / วันทั้งหมดในช่วง (แยกปี) | ความน่าเชื่อถือ |
| `rank_change` | อันดับ `yoy_change_pct` ภายใต้แต่ละวิธี S1–S5 | ทดสอบความไวของข้อสรุป |

**สถานการณ์ทดสอบ:** S1 ตัด NaN เก็บ 0 · S2 ศูนย์ช่วงยาวเป็น NaN แล้วตัด · S3 เฉพาะวันที่ทั้งสองปีมีค่า (paired) · S4 เติม NaN ด้วย median วันเดียวกันในสัปดาห์ของเดือนนั้น · S5 แทน NaN ด้วย 0 (แสดงผลเสีย)
กรณีสำคัญ: ขบ. รถหมวด 3/4 (เป็นศูนย์ทั้งช่วงในปี 2569), ขสมก. รถร่วม (ศูนย์ทั้งสองปี)

In [4]:
# TODO: สร้าง features ตามตารางด้านบน แล้วแสดง .head() และ .describe() เป็นหลักฐาน

## 4. Visualization (อย่างน้อย 2 รูปแบบ)
แนะนำ: ① diverging bar ของ `yoy_change_pct` ② line ค่าเฉลี่ยเคลื่อนที่ 7 วัน ปี 68 vs 69 ③ slope chart/heatmap ของอันดับใน S1–S5

บันทึกทุกกราฟด้วย `prep.save_fig(fig, "<ชื่อ>")` แล้วใช้ไฟล์ใน `figures/` ในรายงานและสไลด์

In [5]:
# TODO: กราฟที่ 1

**การตีความกราฟที่ 1:** _(เขียนจากตัวเลขจริงในกราฟ)_

In [6]:
# TODO: กราฟที่ 2

**การตีความกราฟที่ 2:** _(เขียนจากตัวเลขจริงในกราฟ)_

## 5. Analysis & Findings
- คำตอบของคำถาม:
- ข้อค้นพบสำคัญ:
- ข้อจำกัดของข้อมูล:

## 6. บันทึก Prepared Dataset

In [7]:
# TODO: เปลี่ยน result_df เป็นตารางสุดท้ายที่มี features ใหม่ครบ
# result_df.to_csv(prep.PREPARED_DIR / "q2_yoy_comparison.csv", index=False, encoding="utf-8-sig")